In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("~/ml-projects/stock-ranker/data/features_labeled.csv", parse_dates=["Date"])

print("shape:", df.shape)
print("交易日数量:", df["Date"].nunique())
print("每日股票数:", df.groupby("Date")["ticker"].count().unique())
print("\n前3行：")
print(df.head(3))

shape: (200, 12)
交易日数量: 40
每日股票数: [5]

前3行：
        Date    mom_5d   mom_10d   mom_20d   vol_10d   vol_20d  rsi_dist  \
0 2026-02-12 -0.050506  0.014306  0.007751  0.023757  0.019977 -0.043147   
1 2026-02-13 -0.079464 -0.013337 -0.008484  0.024926  0.020596 -0.575895   
2 2026-02-17 -0.039109 -0.021788  0.033643  0.023383  0.021625  0.125352   

   high_20d_ratio  vol_ratio ticker  next_ret  rank  
0        0.941949   1.273324   AAPL -0.022733     5  
1        0.920536   0.937266   AAPL  0.031668     1  
2        0.949687   0.999233   AAPL  0.001781     5  


In [2]:
feature_cols = ['mom_5d', 'mom_10d', 'mom_20d', 'vol_10d', 'vol_20d',
                'rsi_dist', 'high_20d_ratio', 'vol_ratio']

pairs = []
for date, group in df.groupby("Date"):
    group = group.reset_index(drop=True)
    n = len(group)
    for i in range(n):
        for j in range(i+1, n):
            fi = group.loc[i, feature_cols].values
            fj = group.loc[j, feature_cols].values
            ri = group.loc[i, "rank"]
            rj = group.loc[j, "rank"]
            if ri < rj:       # i 更强，label=1
                pairs.append((*fi, *fj, 1.0))
            elif ri > rj:     # j 更强，label=0
                pairs.append((*fi, *fj, 0.0))
            # ri==rj 理论上不会出现（rank 均匀）

col_names = [f"f_i_{c}" for c in feature_cols] + \
            [f"f_j_{c}" for c in feature_cols] + ["label"]
pair_df = pd.DataFrame(pairs, columns=col_names)

print("pairwise 样本数:", len(pair_df))
print("期望值:", 40 * 10, "（40天 × C(5,2)=10对）")
print("label 分布：")
print(pair_df["label"].value_counts())
print("\n前2行：")
print(pair_df.head(2))

pairwise 样本数: 400
期望值: 400 （40天 × C(5,2)=10对）
label 分布：
label
0.0    220
1.0    180
Name: count, dtype: int64

前2行：
   f_i_mom_5d  f_i_mom_10d  f_i_mom_20d  f_i_vol_10d  f_i_vol_20d  \
0   -0.050506     0.014306     0.007751     0.023757     0.019977   
1   -0.050506     0.014306     0.007751     0.023757     0.019977   

   f_i_rsi_dist  f_i_high_20d_ratio  f_i_vol_ratio  f_j_mom_5d  f_j_mom_10d  \
0     -0.043147            0.941949       1.273324    0.020753    -0.073033   
1     -0.043147            0.941949       1.273324   -0.067170    -0.086475   

   f_j_mom_20d  f_j_vol_10d  f_j_vol_20d  f_j_rsi_dist  f_j_high_20d_ratio  \
0    -0.125256     0.023429     0.030200     -1.179751            0.834333   
1    -0.079919     0.013431     0.013899     -2.223301            0.899066   

   f_j_vol_ratio  label  
0       0.814973    0.0  
1       0.981290    0.0  


In [4]:
from sklearn.model_selection import train_test_split
import os.path

train_df, val_df = train_test_split(pair_df, test_size=0.2, random_state=42)

print("train:", len(train_df), "  val:", len(val_df))
print("train label 分布：")
print(train_df["label"].value_counts())

save_dir = os.path.expanduser("~/ml-projects/stock-ranker/data")
pair_df.to_csv(f"{save_dir}/pairs_train.csv", index=False)
print("\n已保存 pairs_train.csv")

train: 320   val: 80
train label 分布：
label
0.0    175
1.0    145
Name: count, dtype: int64

已保存 pairs_train.csv


In [5]:
import torch
import torch.nn as nn

class RankNet(nn.Module):
    def __init__(self, input_dim=8):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
    
    def forward(self, fi, fj):
        si = self.scorer(fi)  # [B, 1]
        sj = self.scorer(fj)  # [B, 1]
        return torch.sigmoid(si - sj).squeeze(1)  # [B]
    
    def score(self, f):
        # 推理时单独给每只股票打分
        return self.scorer(f).squeeze(1)

model = RankNet(input_dim=8)
print(model)
print("\n参数量:", sum(p.numel() for p in model.parameters()))

# 验证 forward 能跑通
fi_test = torch.randn(4, 8)
fj_test = torch.randn(4, 8)
out = model(fi_test, fj_test)
print("output shape:", out.shape, "  values:", out.detach().numpy().round(3))

RankNet(
  (scorer): Sequential(
    (0): Linear(in_features=8, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): ReLU()
    (5): Linear(in_features=16, out_features=1, bias=True)
  )
)

参数量: 833
output shape: torch.Size([4])   values: [0.533 0.448 0.559 0.458]


In [9]:
from torch.utils.data import DataLoader, TensorDataset

# 准备数据
feat_cols_i = [f"f_i_{c}" for c in feature_cols]
feat_cols_j = [f"f_j_{c}" for c in feature_cols]

def make_tensors(df):
    fi = torch.tensor(df[feat_cols_i].values, dtype=torch.float32)
    fj = torch.tensor(df[feat_cols_j].values, dtype=torch.float32)
    label = torch.tensor(df["label"].values, dtype=torch.float32)
    return fi, fj, label

fi_train, fj_train, y_train = make_tensors(train_df)
fi_val,   fj_val,   y_val   = make_tensors(val_df)

train_loader = DataLoader(TensorDataset(fi_train, fj_train, y_train),
                          batch_size=32, shuffle=True)

# 训练
model = RankNet(input_dim=8)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

for epoch in range(30):
    model.train()
    total_loss = 0
    for fi_b, fj_b, y_b in train_loader:
        optimizer.zero_grad()
        pred = model(fi_b, fj_b)
        loss = criterion(pred, y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        model.eval()
        with torch.no_grad():
            val_pred = model(fi_val, fj_val)
            val_loss = criterion(val_pred, y_val).item()
            val_acc = ((val_pred > 0.5) == y_val.bool()).float().mean().item()
        print(f"Epoch {epoch+1:2d} | train_loss: {total_loss/len(train_loader):.4f} | val_loss: {val_loss:.4f} | val_acc: {val_acc:.4f}")

Epoch  5 | train_loss: 0.6949 | val_loss: 0.6926 | val_acc: 0.5125
Epoch 10 | train_loss: 0.6918 | val_loss: 0.6923 | val_acc: 0.5250
Epoch 15 | train_loss: 0.6920 | val_loss: 0.6922 | val_acc: 0.4750
Epoch 20 | train_loss: 0.6897 | val_loss: 0.6915 | val_acc: 0.5250
Epoch 25 | train_loss: 0.6885 | val_loss: 0.6905 | val_acc: 0.5375
Epoch 30 | train_loss: 0.6860 | val_loss: 0.6882 | val_acc: 0.5250


In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
all_feat_cols = feat_cols_i + feat_cols_j

# 用 train_df fit，transform train 和 val
train_scaled = train_df.copy()
val_scaled = val_df.copy()

train_scaled[all_feat_cols] = scaler.fit_transform(train_df[all_feat_cols])
val_scaled[all_feat_cols]   = scaler.transform(val_df[all_feat_cols])

fi_train, fj_train, y_train = make_tensors(train_scaled)
fi_val,   fj_val,   y_val   = make_tensors(val_scaled)

train_loader = DataLoader(TensorDataset(fi_train, fj_train, y_train),
                          batch_size=32, shuffle=True)

model2 = RankNet(input_dim=8)
optimizer = torch.optim.Adam(model2.parameters(), lr=1e-3)

for epoch in range(30):
    model2.train()
    total_loss = 0
    for fi_b, fj_b, y_b in train_loader:
        optimizer.zero_grad()
        pred = model2(fi_b, fj_b)
        loss = criterion(pred, y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        model2.eval()
        with torch.no_grad():
            val_pred = model2(fi_val, fj_val)
            val_loss = criterion(val_pred, y_val).item()
            val_acc = ((val_pred > 0.5) == y_val.bool()).float().mean().item()
        print(f"Epoch {epoch+1:2d} | train_loss: {total_loss/len(train_loader):.4f} | val_loss: {val_loss:.4f} | val_acc: {val_acc:.4f}")

Epoch  5 | train_loss: 0.6813 | val_loss: 0.6802 | val_acc: 0.5875
Epoch 10 | train_loss: 0.6703 | val_loss: 0.6697 | val_acc: 0.6500
Epoch 15 | train_loss: 0.6460 | val_loss: 0.6599 | val_acc: 0.6625
Epoch 20 | train_loss: 0.6342 | val_loss: 0.6530 | val_acc: 0.6250
Epoch 25 | train_loss: 0.6310 | val_loss: 0.6452 | val_acc: 0.6375
Epoch 30 | train_loss: 0.6130 | val_loss: 0.6396 | val_acc: 0.6500


In [8]:
for epoch in range(31, 100):
    model2.train()
    total_loss = 0
    for fi_b, fj_b, y_b in train_loader:
        optimizer.zero_grad()
        pred = model2(fi_b, fj_b)
        loss = criterion(pred, y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 20 == 0:
        model2.eval()
        with torch.no_grad():
            val_pred = model2(fi_val, fj_val)
            val_loss = criterion(val_pred, y_val).item()
            val_acc = ((val_pred > 0.5) == y_val.bool()).float().mean().item()
        print(f"Epoch {epoch+1:2d} | train_loss: {total_loss/len(train_loader):.4f} | val_loss: {val_loss:.4f} | val_acc: {val_acc:.4f}")

print("训练结束")

Epoch 40 | train_loss: 0.6059 | val_loss: 0.6318 | val_acc: 0.6250
Epoch 60 | train_loss: 0.5690 | val_loss: 0.6121 | val_acc: 0.6375
Epoch 80 | train_loss: 0.5409 | val_loss: 0.5973 | val_acc: 0.6125
Epoch 100 | train_loss: 0.5079 | val_loss: 0.5858 | val_acc: 0.6625
训练结束


In [10]:
import pickle, os

save_dir = os.path.expanduser("~/ml-projects/stock-ranker/data")

# 保存模型
torch.save({
    "model_state": model2.state_dict(),
    "hparams": {"input_dim": 8, "feature_cols": feature_cols}
}, f"{save_dir}/ranknet_v1.pth")

# 保存 scaler
with open(f"{save_dir}/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("已保存：")
print(f"  {save_dir}/ranknet_v1.pth")
print(f"  {save_dir}/scaler.pkl")

# 验证能读回来
ckpt = torch.load(f"{save_dir}/ranknet_v1.pth")
print("\nhparams:", ckpt["hparams"])
print("model_state keys:", list(ckpt["model_state"].keys()))

已保存：
  /Users/tongwenchao/ml-projects/stock-ranker/data/ranknet_v1.pth
  /Users/tongwenchao/ml-projects/stock-ranker/data/scaler.pkl

hparams: {'input_dim': 8, 'feature_cols': ['mom_5d', 'mom_10d', 'mom_20d', 'vol_10d', 'vol_20d', 'rsi_dist', 'high_20d_ratio', 'vol_ratio']}
model_state keys: ['scorer.0.weight', 'scorer.0.bias', 'scorer.3.weight', 'scorer.3.bias', 'scorer.5.weight', 'scorer.5.bias']
